# HookLink2iOS **4.0 / X3** — IPA verificada → Colab → Drive

Este notebook aceita **apenas** IPAs do workflow `.github/workflows/hooklink2ios-v4.yml` com o motor oficial nxnative `ee3ce6a02bc737d38795f0a96cce494cee7a799c` e shaders `b6112b954691`.

Valida SHA-256 do artifact/IPA, relatório de build, motor, chain, iOS 26, Metal, marcador de RomFS externo; baixa e envia em blocos de 64 MiB com retomada.

**Seleciona exclusivamente o artefato do último run 4.0 bem-sucedido; falha se esse artefato não existir, sem fallback para IPAs antigas.**

Execute as três células Python em ordem, no Google Colab. GH_TOKEN precisa de Actions:read. A pasta `ipa` no Drive será reutilizada.


In [ ]:
# HookLink2iOS v4 ONLY: verify exact upstream SHA and rebuild chain, 64 MiB per chunk
REPO="joao9vegeto-stack/KintTany"
BRANCH="HookLink2iOS"
TITLE="0400486024DEC000"
PROJECT_DRIVE_FOLDER="1IynIGg57WnKydeqsdVzOp90tPkiX_eNY"
CHUNK=64*1024*1024
assert CHUNK == 64*1024*1024 and CHUNK%(256*1024)==0
import os,time,re,json,io,hashlib,zipfile,plistlib,shutil,getpass
from pathlib import Path
from urllib.parse import urlparse
import requests
from google.colab import auth,userdata
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
import google.auth
ROOT=Path("/content/HookLink2iOS-transfer");ROOT.mkdir(exist_ok=True)
try:TOKEN=userdata.get("GH_TOKEN") or ""
except Exception:TOKEN=""
if not TOKEN:TOKEN=getpass.getpass("GH_TOKEN (Actions:read para localizar a última IPA): ").strip()
if not TOKEN:raise RuntimeError("GH_TOKEN é obrigatório para localizar artefatos neste repositório")
HEAD={"Accept":"application/vnd.github+json","X-GitHub-Api-Version":"2022-11-28"}
def gitget(path,**opts):
    headers=dict(HEAD)
    if TOKEN:headers["Authorization"]="Bearer "+TOKEN
    r=requests.get("https://api.github.com/repos/"+REPO+path,headers=headers,timeout=40,**opts)
    if r.status_code in (401,403):raise RuntimeError("Configure Colab Secrets GH_TOKEN com permissão Actions read.")
    r.raise_for_status();return r.json()
def sha(path,algo="sha256"):
    h=hashlib.new(algo)
    with open(path,"rb") as f:
        for b in iter(lambda:f.read(CHUNK),b""):h.update(b)
    return h.hexdigest()
auth.authenticate_user()
creds,_=google.auth.default(scopes=["https://www.googleapis.com/auth/drive"])
drive=build("drive","v3",credentials=creds,cache_discovery=False)
runs=gitget("/actions/runs",params={"branch":BRANCH,"status":"success","per_page":100})["workflow_runs"]
V4_WORKFLOW=".github/workflows/hooklink2ios-v4.yml"
runs=[r for r in runs if r.get("head_branch")==BRANCH and r.get("path")==V4_WORKFLOW and r.get("conclusion")=="success"]
runs.sort(key=lambda r:(r.get("created_at",""),int(r.get("id",0))),reverse=True)
if not runs:raise RuntimeError("Nenhum build HookLink 4.0 concluído com sucesso nesta branch. Sem fallback para a versão 1.8.5.")
chosenrun=runs[0] # Somente o último run V4 bem-sucedido: não retroceder para um artifact antigo.
opts=gitget(f"/actions/runs/{chosenrun['id']}/artifacts").get("artifacts",[])
opts=[art for art in opts if not art.get("expired") and art.get("name","").startswith("HookLink2iOS-V4-")
      and int(art.get("size_in_bytes",0))>20*1024*1024
      and (art.get("workflow_run") or {}).get("id",chosenrun["id"])==chosenrun["id"]
      and (art.get("workflow_run") or {}).get("head_sha",chosenrun["head_sha"])==chosenrun["head_sha"]]
if len(opts)!=1:raise RuntimeError(f"Último run V4 {chosenrun['id']} não possui um artefato único e íntegro ({len(opts)} encontrados). Não usar build anterior.")
artifact=opts[0]
print(f"Último build válido: run {chosenrun['id']} / #{chosenrun['run_number']} / SHA {chosenrun['head_sha'][:12]}")
print(f"Artifact: {artifact['name']} (ID {artifact['id']}, {artifact['size_in_bytes']:,} bytes)")
print(f"Transferências: blocos HTTP Range e Drive resumable de {CHUNK//(1024*1024)} MiB.")


In [ ]:
# Download Actions ranged 64 MiB, resume from existing partial bytes, verify each Content-Range.
def download_actions(art):
    global TOKEN
    if not TOKEN:TOKEN=getpass.getpass("GH_TOKEN (Actions read): ").strip()
    meta=gitget("/actions/artifacts/"+str(art["id"]))
    if meta.get("expired"):raise RuntimeError("Artifact expirado. Não há arquivo para baixar.")
    if meta.get("name")!=art.get("name") or (meta.get("workflow_run") or {}).get("id",chosenrun["id"])!=chosenrun["id"]:
        raise RuntimeError("Metadados do artifact não correspondem ao último run 4.0")
    digest=(meta.get("digest") or "")
    if not re.fullmatch(r"sha256:[0-9a-f]{64}",digest):
        raise RuntimeError("GitHub não forneceu um digest SHA-256 válido para o artefato")
    total=int(meta["size_in_bytes"])
    partial=ROOT/f"artifact-{art['id']}.zip.part"
    finished=ROOT/f"artifact-{art['id']}.zip"
    marker=ROOT/f"artifact-{art['id']}.meta.json"
    identity={"id":art["id"],"bytes":total,"digest":meta.get("digest")}
    if finished.exists():
        if finished.stat().st_size==total and sha(finished)==digest.split(":",1)[1]:
            print("Arquivo do último run já baixado, SHA-256 validado; download dispensado.")
            return finished
        finished.unlink()
    if not marker.exists() or json.loads(marker.read_text())!=identity:
        partial.unlink(missing_ok=True);marker.write_text(json.dumps(identity))
    pos=partial.stat().st_size if partial.exists() else 0
    if pos>total:partial.unlink();pos=0
    while pos<total:
        end=min(total,pos+CHUNK)-1
        for attempt in range(8):
            try:
                headers={**HEAD,"Authorization":"Bearer "+TOKEN}
                rr=requests.get(f"https://api.github.com/repos/{REPO}/actions/artifacts/{art['id']}/zip",headers=headers,allow_redirects=False,timeout=40)
                if rr.status_code!=302 or not rr.headers.get("Location"):raise RuntimeError(f"GitHub redirect HTTP {rr.status_code}")
                signed=rr.headers["Location"]
                if urlparse(signed).scheme!="https":raise RuntimeError("Redirect inseguro")
                with requests.get(signed,headers={"Range":f"bytes={pos}-{end}"},stream=True,timeout=(30,180)) as r:
                    m=re.fullmatch(r"bytes\s+(\d+)-(\d+)/(\d+)",r.headers.get("Content-Range",""))
                    if r.status_code!=206 or not m or tuple(map(int,m.groups()))!=(pos,end,total):raise RuntimeError("HTTP Range inválido")
                    received=0
                    with partial.open("r+b" if partial.exists() else "w+b") as f:
                        f.truncate(pos);f.seek(pos)
                        for b in r.iter_content(1024*1024):
                            if b:f.write(b);received+=len(b)
                    if received!=end-pos+1:raise RuntimeError("Chunk incompleto")
                pos=end+1
                print(f"Actions → Colab (Range 64 MiB): {pos/total*100:.1f}% ({pos:,}/{total:,})",end="\r")
                break
            except Exception as e:
                if partial.exists():
                    with partial.open("r+b") as f:f.truncate(pos)
                print("\nChunk retry:",type(e).__name__,str(e))
                if attempt==7:raise
                time.sleep(min(30,2**attempt))
    print()
    partial.replace(finished)
    dg=meta.get("digest") or ""
    if dg.startswith("sha256:") and sha(finished)!=dg.split(":",1)[1]:raise RuntimeError("SHA-256 do artefato GitHub não confere")
    return finished
SOURCE=download_actions(artifact)
origin=f"Actions V4 run {chosenrun['id']} artifact {artifact['id']} (64 MiB chunks)"
with zipfile.ZipFile(SOURCE) as z:
    nested=[i for i in z.infolist() if i.filename.lower().endswith(".ipa") and not i.is_dir()]
    if nested:
        if len(nested)!=1:raise RuntimeError("Mais de uma IPA no artefato")
        IPA=ROOT/Path(nested[0].filename).name
        with z.open(nested[0]) as src,IPA.open("wb") as dest:shutil.copyfileobj(src,dest,CHUNK)
        reports=[i for i in z.infolist() if Path(i.filename).name=="build-report.json"]
        if len(reports)!=1:raise RuntimeError("Artifact v4 não contém relatório único")
        report=json.loads(z.read(reports[0]))
        if report.get("hooklink_release")!="4.0":raise RuntimeError("Não é HookLink 4.0")
        if report.get("engine_commit")!="ee3ce6a02bc737d38795f0a96cce494cee7a799c":raise RuntimeError("Engine nxnative X3 divergente")
        if report.get("shader_chain")!="b6112b954691":raise RuntimeError("Biblioteca Metal não é da cadeia X3")
        if report.get("original_1_8_shader_image_reused") is not False:raise RuntimeError("Bibliotecas antigas não descartadas")
        if report.get("romfs_mode")!="separate" or report.get("gpu_arch")!="none":raise RuntimeError("Port X3 não é IPA thin/GPU none")
        if int(report.get("ipa",{}).get("size",0))!=IPA.stat().st_size:raise RuntimeError("Tamanho IPA divergente do relatório")
        if report.get("ipa",{}).get("sha256")!=sha(IPA):raise RuntimeError("SHA-256 IPA divergente do relatório")
        if True:
            want=report.get("ipa",{}).get("sha256")
            if want and sha(IPA)!=want:raise RuntimeError("SHA-256 divergente do build-report")
            if report.get("output_size") is not None and int(report["output_size"])!=IPA.stat().st_size:raise RuntimeError("IPA diverge do tamanho registrado no repack")
            if report.get("output_sha256") and sha(IPA)!=report["output_sha256"]:raise RuntimeError("IPA diverge do SHA-256 registrado no repack")
    else:IPA=SOURCE
if IPA.stat().st_size>=4_000_000_000:raise RuntimeError("IPA excede 4 GB; rejeitada antes do upload para evitar ZIP64/erro de instalação")
with zipfile.ZipFile(IPA) as z:
    names=z.namelist();pre="Payload/DmIOS.app/"
    if len(names)!=len(set(names)) or any(pre+x not in names for x in ("DmIOS","romfs.id","Info.plist")):raise RuntimeError("IPA incompleta")
    marker=z.read(pre+"romfs.id").decode("utf8","replace")
    if "title="+TITLE not in marker or "build=ee3ce6a02bc737d38795f0a96cce494cee7a799c" not in marker:raise RuntimeError("Marcador de RomFS não é do X3")
    if str(plistlib.loads(z.read(pre+"Info.plist")).get("MinimumOSVersion"))!="26.0":raise RuntimeError("iOS target incorreto")
    if not any(re.fullmatch(r"Payload/DmIOS\.app/msl/b-[^/]+\.img(?:\.[0-9]+)?", n) for n in names):raise RuntimeError("Metal image/parts ausentes")
    if any(n.lower().endswith("romfs.img") for n in names):raise RuntimeError("RomFS deveria ser externo")
    if z.testzip():raise RuntimeError("ZIP CRC inválido")
IPA_HASH=sha(IPA)
print("IPA verificada:",IPA.name,IPA.stat().st_size,"bytes, SHA-256:",IPA_HASH)


In [ ]:
# Upload Colab → Drive resumable 64 MiB + MD5/SHA-256 readback.
def drive_folder():
    parent=PROJECT_DRIVE_FOLDER
    try:drive.files().get(fileId=parent,fields="id",supportsAllDrives=True).execute()
    except Exception:
        found=drive.files().list(q="name='HookLink2iOS' and mimeType='application/vnd.google-apps.folder' and trashed=false",fields="files(id)",pageSize=20).execute().get("files",[])
        parent=found[0]["id"] if found else drive.files().create(body={"name":"HookLink2iOS","mimeType":"application/vnd.google-apps.folder"},fields="id").execute()["id"]
    found=drive.files().list(q=f"'{parent}' in parents and name='ipa' and mimeType='application/vnd.google-apps.folder' and trashed=false",fields="files(id)",pageSize=20).execute().get("files",[])
    return found[0]["id"] if found else drive.files().create(body={"name":"ipa","mimeType":"application/vnd.google-apps.folder","parents":[parent]},fields="id").execute()["id"]
folder=drive_folder()
name=IPA.stem+"-"+IPA_HASH[:12]+".ipa"
queryname=name.replace("'","\\'")
existing=drive.files().list(q=f"'{folder}' in parents and name='{queryname}' and trashed=false",fields="files(id,name,size,md5Checksum,appProperties,webViewLink)",pageSize=100).execute().get("files",[])
MD5=sha(IPA,"md5")
current=next((x for x in existing if int(x.get("size",0))==IPA.stat().st_size and x.get("md5Checksum")==MD5 and (x.get("appProperties") or {}).get("sha256")==IPA_HASH),None)
if current:
    print("Cópia idêntica encontrada no Drive; upload desnecessário.")
else:
    upload=MediaFileUpload(str(IPA),mimetype="application/octet-stream",chunksize=CHUNK,resumable=True)
    req=drive.files().create(body={"name":name,"parents":[folder],"appProperties":{"sha256":IPA_HASH,"titleId":TITLE,"branch":BRANCH,"engine":"ee3ce6a02bc737d38795f0a96cce494cee7a799c","hooklink":"4.0"}},media_body=upload,fields="id,name,size,md5Checksum,appProperties,webViewLink")
    current=None
    while current is None:
        for attempt in range(8):
            try:
                status,current=req.next_chunk()
                if status:print(f"Colab → Drive (resumable 64 MiB): {status.progress()*100:.1f}%",end="\r")
                break
            except Exception as e:
                print("\nRetry upload chunk:",type(e).__name__)
                if attempt==7:raise
                time.sleep(min(30,2**attempt))
    print()
check=drive.files().get(fileId=current["id"],fields="id,name,size,md5Checksum,appProperties,webViewLink",supportsAllDrives=True).execute()
if int(check.get("size",0))!=IPA.stat().st_size or check.get("md5Checksum")!=MD5 or (check.get("appProperties") or {}).get("sha256")!=IPA_HASH:
    raise RuntimeError("Drive readback não validou tamanho, MD5 e SHA-256")
print("TRANSFERÊNCIA INTEGRALMENTE VALIDADA")
print("Origem:",origin)
print("Nome:",check["name"])
print("SHA-256:",IPA_HASH)
print("Drive:",check.get("webViewLink") or "https://drive.google.com/file/d/"+check["id"]+"/view")
